Imports and project paths

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import CalibrationDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Project folders:
# Expected structure:
# PPD_Risk_Prediction/
#   dataset/post natal data.csv
#   notebooks/04_model_tuning.ipynb
#   models/
#   outputs/metrics/
#   outputs/graphs/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUT_DIR / "metrics"
GRAPHS_DIR = OUTPUT_DIR / "graphs"

for folder in [MODEL_DIR, METRICS_DIR, GRAPHS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = [
    DATA_DIR / "post natal data.csv",
    DATA_DIR / "raw" / "post natal data.csv",
]
DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Put 'post natal data.csv' in either "
        "'dataset/' or 'dataset/raw/' inside the project folder. "
        f"Checked: {DATA_PATH_CANDIDATES}"
    )

TARGET_COLUMN = "Feeling anxious"
TARGET_MAPPING = {"no": 0, "yes": 1}

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Target:", TARGET_COLUMN)

Project root: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction
Dataset: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction\dataset\raw\post natal data.csv
Target: Feeling anxious


Load the candidate model and metadata

In [2]:
MODEL_PATH = MODEL_DIR / "best_ppd_model.joblib"
METADATA_PATH = MODEL_DIR / "model_metadata.json"

if not MODEL_PATH.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Run Notebook 05 first to create best_ppd_model.joblib and model_metadata.json."
    )

candidate_model = joblib.load(MODEL_PATH)
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

feature_columns = metadata["input_features"]

print("Candidate model:", metadata["model_name"])
print("Target:", metadata["target_column"])
print("Expected input fields:", feature_columns)
print("\nWARNING:", metadata["warning"])

Candidate model: SVM
Target: Feeling anxious
Expected input fields: ['Age', 'Feeling sad or Tearful', 'Irritable towards baby & partner', 'Trouble sleeping at night', 'Problems concentrating or making decision', 'Overeating or loss of appetite', 'Feeling of guilt', 'Problems of bonding with baby', 'Suicide attempt']



Load the original dataset to show valid example response values

In [3]:
df_reference = pd.read_csv(DATA_PATH)
df_reference.columns = df_reference.columns.astype(str).str.strip()

for col in feature_columns:
    if col not in df_reference.columns:
        raise KeyError(f"Feature '{col}' is missing from the dataset.")

for col in feature_columns:
    values = df_reference[col].dropna().astype(str).str.strip().unique().tolist()
    print(f"\n{col}")
    print("Example values:", values[:15])


Age
Example values: ['35-40', '40-45', '30-35', '45-50', '25-30']

Feeling sad or Tearful
Example values: ['Yes', 'No', 'Sometimes']

Irritable towards baby & partner
Example values: ['Yes', 'No', 'Sometimes']

Trouble sleeping at night
Example values: ['Two or more days a week', 'No', 'Yes']

Problems concentrating or making decision
Example values: ['Yes', 'No', 'Often']

Overeating or loss of appetite
Example values: ['Yes', 'No', 'Not at all']

Feeling of guilt
Example values: ['No', 'Yes', 'Maybe']

Problems of bonding with baby
Example values: ['Yes', 'Sometimes', 'No']

Suicide attempt
Example values: ['Yes', 'No', 'Not interested to say']


Create an example input

In [4]:
# Placeholder example only. Replace values with valid questionnaire responses.
example_input = {}

for feature in feature_columns:
    available_values = (
        df_reference[feature]
        .dropna()
        .astype(str)
        .str.strip()
    )
    if available_values.empty:
        example_input[feature] = "Unknown"
    else:
        # Use the most common observed value as a neutral technical example.
        example_input[feature] = available_values.mode().iloc[0]

input_df = pd.DataFrame([example_input], columns=feature_columns)
display(input_df)

,Age,Feeling sad or Tearful,Irritable towards baby & partner,Trouble sleeping at night,Problems concentrating or making decision,Overeating or loss of appetite,Feeling of guilt,Problems of bonding with baby,Suicide attempt
0,40-45,Yes,Yes,Two or more days a week,No,No,No,No,No


Generate a proxy probability demonstration

In [5]:
proxy_probability = float(candidate_model.predict_proba(input_df)[0, 1])
proxy_class = int(candidate_model.predict(input_df)[0])

print(f"Predicted proxy class: {proxy_class}")
print(f"Predicted probability for proxy class 'Yes': {proxy_probability:.4f}")
print("\nThis is NOT a PPD risk probability or diagnosis.")

Predicted proxy class: 1
Predicted probability for proxy class 'Yes': 0.9973

This is NOT a PPD risk probability or diagnosis.


Optional: demonstrate score bands (NOT clinical categories)

In [6]:
def proxy_score_band(probability):
    if probability < 0.33:
        return "Lower proxy score band (demo only)"
    elif probability < 0.66:
        return "Middle proxy score band (demo only)"
    return "Higher proxy score band (demo only)"

print("Demonstration band:", proxy_score_band(proxy_probability))
print("Reminder: These bands are arbitrary and have no clinical meaning.")

Demonstration band: Higher proxy score band (demo only)
Reminder: These bands are arbitrary and have no clinical meaning.


Save the demonstration output

In [7]:
demo_output = {
    "model_name": metadata["model_name"],
    "target_column": metadata["target_column"],
    "predicted_proxy_class": proxy_class,
    "predicted_proxy_probability": proxy_probability,
    "demonstration_band": proxy_score_band(proxy_probability),
    "warning": metadata["warning"]
}

demo_path = OUTPUT_DIR / "example_proxy_prediction.json"
with open(demo_path, "w", encoding="utf-8") as f:
    json.dump(demo_output, f, indent=4)

print("Saved demonstration output to:", demo_path)

Saved demonstration output to: c:\Users\user\Desktop\Care AI\PPD_Risk_Prediction\outputs\example_proxy_prediction.json
